In [7]:
import os
import numpy as np
import pandas as pd

RUN_TAG = "251110"
RESULTS_DIR = f"out/{RUN_TAG}/results"
DESC_FILE = os.path.join(RESULTS_DIR, "FIFO_EXP_path_descriptors_machine_and_activity_level_STABLE.csv")

# Choose which entropy you want to use for the 3x3 axis
ENTROPY_COL = "H_norm"          # try also: "H2_norm_paths"
D_COL = "D_paths"

# Proxies used ONLY to judge whether bins are meaningfully different
PROXIES = ["var_total", "K_paths", "wip_max", "q_max"]

MIN_CELL = 1  # require at least this many logs in each of the 9 cells

# Candidate manual thresholds to try (add/remove as you like)
D_candidates = [
    (0.15, 0.45),
    (0.20, 0.50),
    (0.25, 0.55),
    (0.10, 0.35),
    (0.30, 0.70),
    (0.35, 0.80),
    (0.15, 0.50)
]

H_candidates = [
    (0.82, 0.92),
    (0.85, 0.95),
    (0.90, 0.97),
    (0.80, 0.90),
    (0.85, 0.97),
    (0.85, 0.98)
]

def cut3(x: pd.Series, a: float, b: float) -> pd.Categorical:
    # low <= a, mid (a,b], high > b
    return pd.cut(
        x,
        bins=[-np.inf, a, b, np.inf],
        labels=["low", "mid", "high"],
        include_lowest=True
    )

def cell_stats(tmp: pd.DataFrame, proxies: list[str]) -> dict:
    """
    Returns:
      - mat: 3x3 counts table
      - min_cell: smallest cell count in the 3x3
      - spreads_raw: dict proxy -> (max_cell_mean - min_cell_mean)
      - spreads_norm: dict proxy -> normalized spread (divided by overall std of that proxy)
      - score: sum of normalized spreads (higher = more separation)
    """
    mat = pd.crosstab(tmp["H_bin"], tmp["D_bin"], dropna=False)

    # If any cell missing from crosstab indexing, ensure full 3x3 shape
    mat = mat.reindex(index=["low","mid","high"], columns=["low","mid","high"], fill_value=0)
    min_cell = int(mat.min().min())

    # compute cell means for each proxy
    grp = tmp.groupby(["H_bin", "D_bin"], observed=True)[proxies].mean()
    # ensure full 3x3 exists for mean table too (missing cells become NaN)
    grp = grp.reindex(pd.MultiIndex.from_product([["low","mid","high"], ["low","mid","high"]]))

    spreads_raw = {}
    spreads_norm = {}

    for p in proxies:
        cell_means = grp[p]

        # If there are NaNs (empty cells), spread isn't meaningful
        if cell_means.isna().any():
            spreads_raw[p] = np.nan
            spreads_norm[p] = np.nan
            continue

        spread = float(cell_means.max() - cell_means.min())
        spreads_raw[p] = spread

        sd = float(tmp[p].std(ddof=0))  # population std
        spreads_norm[p] = (spread / sd) if sd > 1e-12 else 0.0

    # Combined score: sum of normalized spreads (ignore NaNs)
    score = float(np.nansum(list(spreads_norm.values())))

    return {
        "mat": mat,
        "min_cell": min_cell,
        "spreads_raw": spreads_raw,
        "spreads_norm": spreads_norm,
        "score": score,
    }

def main():
    df = pd.read_csv(DESC_FILE)
    df = df.replace([np.inf, -np.inf], np.nan)

    need = [D_COL, ENTROPY_COL] + PROXIES + ["log_name"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in descriptor file: {missing}")

    df = df.dropna(subset=need).copy()

    results = []

    for (d1, d2) in D_candidates:
        for (h1, h2) in H_candidates:
            tmp = df.copy()
            tmp["D_bin"] = cut3(tmp[D_COL], d1, d2)
            tmp["H_bin"] = cut3(tmp[ENTROPY_COL], h1, h2)

            stats = cell_stats(tmp, PROXIES)

            ok = (stats["min_cell"] >= MIN_CELL) and np.isfinite(stats["score"])
            if not ok:
                continue

            row = {
                "D_edges": (d1, d2),
                "H_edges": (h1, h2),
                "min_cell": stats["min_cell"],
                "score": stats["score"],
            }

            # keep both raw and normalized spreads (normalized is what score uses)
            for p in PROXIES:
                row[f"spread_raw_{p}"] = stats["spreads_raw"][p]
                row[f"spread_norm_{p}"] = stats["spreads_norm"][p]

            # store mat as string for quick peek later
            row["counts_3x3"] = stats["mat"].to_string()

            results.append(row)

    if not results:
        print("No candidate binning satisfied MIN_CELL. Lower MIN_CELL or widen candidate thresholds.")
        return

    res = pd.DataFrame(results).sort_values(["score", "min_cell"], ascending=[False, False])

    # Print top 5 candidates with per-proxy spreads
    show_cols = ["D_edges", "H_edges", "min_cell", "score"] + \
                [f"spread_norm_{p}" for p in PROXIES] + \
                [f"spread_raw_{p}" for p in PROXIES]

    print("\n=== Top 5 binning schemes (ranked by score) ===")
    print(res[show_cols].head(5).to_string(index=False))

    # Print the 3x3 counts for the best scheme
    best = res.iloc[0]
    print("\n=== Best scheme 3x3 counts ===")
    print(best["counts_3x3"])

    # Optional: save ranking table
    out_rank = os.path.join(RESULTS_DIR, f"FIFO_EXP_rank_binnings_{ENTROPY_COL}_vs_{D_COL}.csv")
    res.drop(columns=["counts_3x3"]).to_csv(out_rank, index=False)
    print(f"\nSaved ranking to: {out_rank}")

if __name__ == "__main__":
    main()



=== Top 5 binning schemes (ranked by score) ===
     D_edges      H_edges  min_cell     score  spread_norm_var_total  spread_norm_K_paths  spread_norm_wip_max  spread_norm_q_max  spread_raw_var_total  spread_raw_K_paths  spread_raw_wip_max  spread_raw_q_max
(0.15, 0.45) (0.85, 0.95)         1 15.711325               5.476349             2.451144             3.858041           3.925791          17764.509270         5252.893939           48.166667         47.772727
 (0.1, 0.35) (0.85, 0.95)         1 13.793234               4.137299             2.170862             3.951488           3.533585          13420.819720         4652.238095           49.333333         43.000000
(0.15, 0.45) (0.85, 0.97)         1  7.998591               1.935865             2.744313             1.606405           1.712008           6279.674008         5881.166667           20.055556         20.833333
(0.15, 0.45) (0.85, 0.98)         1  7.998591               1.935865             2.744313             1.606405 

In [9]:
import os
import numpy as np
import pandas as pd

RUN_TAG = "251110"
RESULTS_DIR = f"out/{RUN_TAG}/results"
DESC_FILE = os.path.join(RESULTS_DIR, "FIFO_EXP_path_descriptors_machine_and_activity_level_STABLE.csv")

ENTROPY_COL = "H_norm"   # or "H2_norm_paths"
D_COL = "D_paths"
PROXIES = ["var_total", "K_paths", "wip_max", "q_max"]

# ----- put your proxy-optimized best here -----
PROXY_OPT_D_EDGES = (0.15, 0.45)
PROXY_OPT_H_EDGES = (0.85, 0.95)

Q_LOW, Q_HIGH = (1/3, 2/3)

def cut3(x: pd.Series, a: float, b: float) -> pd.Categorical:
    return pd.cut(
        x.astype(float),
        bins=[-np.inf, a, b, np.inf],
        labels=["low", "mid", "high"],
        include_lowest=True
    )

def cell_stats(tmp: pd.DataFrame, proxies: list[str]) -> dict:
    mat = pd.crosstab(tmp["H_bin"], tmp["D_bin"], dropna=False)
    mat = mat.reindex(index=["low","mid","high"], columns=["low","mid","high"], fill_value=0)
    min_cell = int(mat.min().min())

    grp = tmp.groupby(["H_bin", "D_bin"], observed=True)[proxies].mean()
    grp = grp.reindex(pd.MultiIndex.from_product([["low","mid","high"], ["low","mid","high"]]))

    spreads_raw, spreads_norm = {}, {}
    for p in proxies:
        cell_means = grp[p]
        if cell_means.isna().any():
            spreads_raw[p] = np.nan
            spreads_norm[p] = np.nan
            continue
        spread = float(cell_means.max() - cell_means.min())
        spreads_raw[p] = spread
        sd = float(tmp[p].std(ddof=0))
        spreads_norm[p] = (spread / sd) if sd > 1e-12 else 0.0

    score = float(np.nansum(list(spreads_norm.values())))

    return {
        "mat": mat,
        "min_cell": min_cell,
        "spreads_raw": spreads_raw,
        "spreads_norm": spreads_norm,
        "score": score,
        "cell_means": grp,
    }

def evaluate_scheme(df: pd.DataFrame, d_edges: tuple[float,float], h_edges: tuple[float,float], name: str):
    tmp = df.copy()
    tmp["D_bin"] = cut3(tmp[D_COL], d_edges[0], d_edges[1])
    tmp["H_bin"] = cut3(tmp[ENTROPY_COL], h_edges[0], h_edges[1])
    st = cell_stats(tmp, PROXIES)

    print("\n" + "="*60)
    print(f"{name}")
    print(f"D_edges={d_edges} | H_edges={h_edges}")
    print(f"min_cell={st['min_cell']} | proxy_score(sum norm spreads)={st['score']:.4f}")
    print("\n3x3 counts:")
    print(st["mat"])

    print("\nProxy spreads (normalized):")
    for p in PROXIES:
        print(f"  {p:10s}: {st['spreads_norm'][p]:.4f}  (raw={st['spreads_raw'][p]:.4f})")

    return st

def main():
    df = pd.read_csv(DESC_FILE).replace([np.inf, -np.inf], np.nan)

    need = ["log_name", D_COL, ENTROPY_COL] + PROXIES
    df = df.dropna(subset=need).copy()

    # quantile cutpoints (ONLY D and H)
    d1 = float(df[D_COL].quantile(Q_LOW))
    d2 = float(df[D_COL].quantile(Q_HIGH))
    h1 = float(df[ENTROPY_COL].quantile(Q_LOW))
    h2 = float(df[ENTROPY_COL].quantile(Q_HIGH))

    print("Quantile cutpoints:")
    print(f"  {D_COL}: {d1:.6f}, {d2:.6f}")
    print(f"  {ENTROPY_COL}: {h1:.6f}, {h2:.6f}")

    st_quant = evaluate_scheme(df, (d1, d2), (h1, h2), name="SCHEME A: Quantile cutpoints (hypothesis-clean)")
    st_proxy = evaluate_scheme(df, PROXY_OPT_D_EDGES, PROXY_OPT_H_EDGES, name="SCHEME B: Proxy-optimized cutpoints (diagnostic benchmark)")

    print("\n" + "-"*60)
    print("Quick takeaway:")
    print("- If quantile scheme has much lower proxy_score, it just means the bins are less separated on congestion/variance.")
    print("- That does NOT invalidate it; it just tells you whether your 3×3 bins correspond to distinct regimes.")
    print("- For publication later, you’ll likely want more logs per cell anyway.")

if __name__ == "__main__":
    main()


Quantile cutpoints:
  D_paths: 0.193191, 0.443209
  H_norm: 0.891891, 0.992661

SCHEME A: Quantile cutpoints (hypothesis-clean)
D_edges=(0.19319050585114697, 0.4432092173672492) | H_edges=(0.891890861020497, 0.992661313030653)
min_cell=2 | proxy_score(sum norm spreads)=9.4296

3x3 counts:
D_bin  low  mid  high
H_bin                
low     13   13     2
mid      8   12     8
high     7    3    18

Proxy spreads (normalized):
  var_total : 2.3918  (raw=7758.5605)
  K_paths   : 2.7399  (raw=5871.6429)
  wip_max   : 2.2127  (raw=27.6250)
  q_max     : 2.0852  (raw=25.3750)

SCHEME B: Proxy-optimized cutpoints (diagnostic benchmark)
D_edges=(0.15, 0.45) | H_edges=(0.85, 0.95)
min_cell=1 | proxy_score(sum norm spreads)=15.7113

3x3 counts:
D_bin  low  mid  high
H_bin                
low      1   13     1
mid      6    9     2
high     9   21    22

Proxy spreads (normalized):
  var_total : 5.4763  (raw=17764.5093)
  K_paths   : 2.4511  (raw=5252.8939)
  wip_max   : 3.8580  (raw=48.1667)
  q

In [10]:
import os
import numpy as np
import pandas as pd

"""
Allocate STABLE MuProMAC logs into a 3x3 matrix using ONLY:
  - D_paths (between-path variance share)
  - One chosen entropy column (H_norm / H2_norm_paths / H_norm_act)

It will:
  1) Compute quantile cutpoints (q=1/3 and q=2/3) for BOTH axes (hypothesis-clean)
  2) Assign L/M/H bins for each log on both axes
  3) Create cell_id (1..9) and cell_label (e.g., E_H__D_M)
  4) Print 3x3 counts and proxy diagnostics (var_total, K_paths, wip_max, q_max)
  5) Also evaluate a "proxy-optimized" benchmark scheme (optional, for comparison only)
  6) Save per-log allocations to CSV for BOTH schemes

How to use:
  - Run as-is (default ENTROPY_COL="H_norm")
  - Change ENTROPY_COL to "H2_norm_paths" or "H_norm_act" and rerun
"""

RUN_TAG = "251110"
RESULTS_DIR = f"out/{RUN_TAG}/results"
DESC_FILE = os.path.join(RESULTS_DIR, "FIFO_EXP_path_descriptors_machine_and_activity_level_STABLE.csv")

# ----- choose entropy axis here -----
ENTROPY_COL = "H_norm"          # try also: "H2_norm_paths", "H_norm_act"
D_COL = "D_paths"

# Diagnostics only (NOT used to define bins)
PROXIES = ["var_total", "K_paths", "wip_max", "q_max"]

# Quantile cutpoints for hypothesis-clean binning
Q_LOW, Q_HIGH = (1/3, 2/3)

# Optional: benchmark scheme you found by proxy optimization (diagnostic comparison)
# Set to None if you don't want this comparison.
PROXY_OPT_D_EDGES = (0.15, 0.45)
PROXY_OPT_H_EDGES = (0.85, 0.95)

# Output paths
OUT_QUANT_ALLOC = os.path.join(RESULTS_DIR, f"FIFO_EXP_alloc_3x3_QUANT_{ENTROPY_COL}_vs_{D_COL}.csv")
OUT_PROXY_ALLOC = os.path.join(RESULTS_DIR, f"FIFO_EXP_alloc_3x3_PROXYOPT_{ENTROPY_COL}_vs_{D_COL}.csv")


def cut3_LMH(x: pd.Series, a: float, b: float) -> pd.Categorical:
    """L <= a, M (a,b], H > b"""
    return pd.cut(
        x.astype(float),
        bins=[-np.inf, a, b, np.inf],
        labels=["L", "M", "H"],
        include_lowest=True
    )

def cell_id(e_bin: str, d_bin: str) -> int:
    """Row-major: E(L/M/H) x D(L/M/H) -> 1..9"""
    e_map = {"L": 0, "M": 1, "H": 2}
    d_map = {"L": 0, "M": 1, "H": 2}
    return 3 * e_map[e_bin] + d_map[d_bin] + 1

def compute_proxy_diagnostics(tmp: pd.DataFrame) -> dict:
    """
    For each proxy:
      - compute cell mean table over 3x3
      - spread_raw = max(mean_cell) - min(mean_cell) across the 9 cells (only if all cells non-empty)
      - spread_norm = spread_raw / overall std(proxy)
    """
    # counts
    mat = pd.crosstab(tmp["E_bin"], tmp["D_bin"], dropna=False)
    mat = mat.reindex(index=["L","M","H"], columns=["L","M","H"], fill_value=0)

    # cell means
    grp = tmp.groupby(["E_bin","D_bin"], observed=True)[PROXIES].mean()
    grp = grp.reindex(pd.MultiIndex.from_product([["L","M","H"], ["L","M","H"]]))

    spreads_raw, spreads_norm = {}, {}
    for p in PROXIES:
        cm = grp[p]
        if cm.isna().any():  # empty cells => can't define spread over 9 cells
            spreads_raw[p] = np.nan
            spreads_norm[p] = np.nan
            continue
        spread = float(cm.max() - cm.min())
        spreads_raw[p] = spread
        sd = float(tmp[p].std(ddof=0))
        spreads_norm[p] = (spread / sd) if sd > 1e-12 else 0.0

    score = float(np.nansum(list(spreads_norm.values())))
    return {
        "counts": mat,
        "cell_means": grp,
        "spreads_raw": spreads_raw,
        "spreads_norm": spreads_norm,
        "score": score,
        "min_cell": int(mat.min().min()),
    }

def allocate(df: pd.DataFrame, d_edges: tuple[float,float], e_edges: tuple[float,float], scheme_name: str) -> pd.DataFrame:
    """Assign bins + cell ids."""
    tmp = df.copy()
    d1, d2 = d_edges
    e1, e2 = e_edges

    tmp["D_cut1"], tmp["D_cut2"] = float(d1), float(d2)
    tmp["E_cut1"], tmp["E_cut2"] = float(e1), float(e2)

    tmp["D_bin"] = cut3_LMH(tmp[D_COL], d1, d2)
    tmp["E_bin"] = cut3_LMH(tmp[ENTROPY_COL], e1, e2)

    tmp["cell_id"] = [cell_id(e, d) for e, d in zip(tmp["E_bin"].astype(str), tmp["D_bin"].astype(str))]
    tmp["cell_label"] = "E_" + tmp["E_bin"].astype(str) + "__D_" + tmp["D_bin"].astype(str)

    tmp["scheme"] = scheme_name
    tmp["entropy_col"] = ENTROPY_COL
    tmp["d_col"] = D_COL

    return tmp

def print_scheme_report(tmp: pd.DataFrame, scheme_name: str):
    diag = compute_proxy_diagnostics(tmp)

    print("\n" + "="*70)
    print(f"{scheme_name}")
    print(f"Entropy axis: {ENTROPY_COL} | D axis: {D_COL}")
    print(f"D cutpoints: {tmp['D_cut1'].iloc[0]:.6f}, {tmp['D_cut2'].iloc[0]:.6f}")
    print(f"E cutpoints: {tmp['E_cut1'].iloc[0]:.6f}, {tmp['E_cut2'].iloc[0]:.6f}")
    print(f"min_cell={diag['min_cell']} | proxy_score(sum norm spreads)={diag['score']:.4f}")

    print("\n3x3 counts (E rows x D cols):")
    print(diag["counts"])

    print("\nProxy spreads (normalized) [diagnostic only]:")
    for p in PROXIES:
        print(f"  {p:10s}: {diag['spreads_norm'][p]:.4f}  (raw={diag['spreads_raw'][p]:.4f})")

def main():
    if not os.path.isfile(DESC_FILE):
        raise FileNotFoundError(f"Descriptor file not found: {DESC_FILE}")

    df = pd.read_csv(DESC_FILE).replace([np.inf, -np.inf], np.nan)

    need = ["log_name", D_COL, ENTROPY_COL] + PROXIES
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in descriptor file: {missing}")

    df = df.dropna(subset=need).copy()

    # ----------------------------
    # SCHEME A: Quantile cutpoints
    # ----------------------------
    d1 = float(df[D_COL].quantile(Q_LOW))
    d2 = float(df[D_COL].quantile(Q_HIGH))
    e1 = float(df[ENTROPY_COL].quantile(Q_LOW))
    e2 = float(df[ENTROPY_COL].quantile(Q_HIGH))

    quant = allocate(df, (d1, d2), (e1, e2), scheme_name="QUANTILES_Q33_Q66")
    print_scheme_report(quant, "SCHEME A: Quantile cutpoints (hypothesis-clean)")

    # Save quant allocation
    quant_cols = ["log_name", D_COL, ENTROPY_COL, "E_bin", "D_bin", "cell_id", "cell_label",
                  "D_cut1", "D_cut2", "E_cut1", "E_cut2", "scheme", "entropy_col", "d_col"]
    quant[quant_cols].to_csv(OUT_QUANT_ALLOC, index=False)
    print(f"\nSaved quantile allocations to: {OUT_QUANT_ALLOC}")

    # -------------------------------------------
    # SCHEME B: Proxy-optimized benchmark (optional)
    # -------------------------------------------
    if PROXY_OPT_D_EDGES is not None and PROXY_OPT_H_EDGES is not None:
        proxyopt = allocate(df, PROXY_OPT_D_EDGES, PROXY_OPT_H_EDGES, scheme_name="PROXY_OPT_BENCHMARK")
        print_scheme_report(proxyopt, "SCHEME B: Proxy-optimized cutpoints (diagnostic benchmark)")

        proxyopt[quant_cols].to_csv(OUT_PROXY_ALLOC, index=False)
        print(f"\nSaved proxy-optimized allocations to: {OUT_PROXY_ALLOC}")
    else:
        print("\nSkipping proxy-optimized benchmark (edges set to None).")


if __name__ == "__main__":
    main()



SCHEME A: Quantile cutpoints (hypothesis-clean)
Entropy axis: H_norm | D axis: D_paths
D cutpoints: 0.193191, 0.443209
E cutpoints: 0.891891, 0.992661
min_cell=2 | proxy_score(sum norm spreads)=9.4296

3x3 counts (E rows x D cols):
D_bin   L   M   H
E_bin            
L      13  13   2
M       8  12   8
H       7   3  18

Proxy spreads (normalized) [diagnostic only]:
  var_total : 2.3918  (raw=7758.5605)
  K_paths   : 2.7399  (raw=5871.6429)
  wip_max   : 2.2127  (raw=27.6250)
  q_max     : 2.0852  (raw=25.3750)

Saved quantile allocations to: out/251110/results\FIFO_EXP_alloc_3x3_QUANT_H_norm_vs_D_paths.csv

SCHEME B: Proxy-optimized cutpoints (diagnostic benchmark)
Entropy axis: H_norm | D axis: D_paths
D cutpoints: 0.150000, 0.450000
E cutpoints: 0.850000, 0.950000
min_cell=1 | proxy_score(sum norm spreads)=15.7113

3x3 counts (E rows x D cols):
D_bin  L   M   H
E_bin           
L      1  13   1
M      6   9   2
H      9  21  22

Proxy spreads (normalized) [diagnostic only]:
  var_t